In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ================================================================
# mistral_vuln_optimized.py
# Mistral-7B-Instruct-v0.1 — PrimeVul vulnerability classification
#
# Key improvements over baseline:
#   1. Correct Mistral [INST] template (no <<SYS>> — Mistral != Llama)
#   2. Unbiased YES/NO prompt (no SAFE/VULNERABLE in prompt body)
#   3. Code preprocessing: strip comments, includes, macros
#   4. bfloat16 precision (safer than float16 on Kaggle T4/P100)
#   5. Logit-level scoring — reads token log-probabilities directly,
#      giving calibrated confidence scores instead of free text
#   6. Majority-vote fallback when logit scoring fails
#   7. MCC + confusion matrix (primary metrics for imbalanced binary)
#   8. Threshold sweep diagnostic to maximise MCC post-run
#   9. Incremental CSV saves every BATCH_SAVE rows
# ================================================================

# !pip install transformers accelerate scikit-learn tqdm pandas -q

import torch
import re
import json
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix,
)

# ================================================================
# CONFIG — edit here only
# ================================================================
MODEL_NAME    = "mistralai/Mistral-7B-Instruct-v0.1"
MAX_CODE_TOKS = 512      # tokens for code; total prompt stays under 1024
MAX_NEW_TOKS  = 20       # Mistral-7B can be verbose; 20 is safe
VOTE_ROUNDS   = 3        # odd → no tie; round 0 greedy, rest sampled
VOTE_TEMP     = 0.6      # temperature for sampled rounds
LOGIT_MODE    = True     # True = score via token log-odds (recommended)
LOGIT_THRESH  = 0.0      # log-odds cut-off: >0 → YES (vulnerable)
N_SAMPLES     = None     # None = full dataset; set int for quick debug
BATCH_SAVE    = 50
OUT_DIR       = "/kaggle/working"

# ================================================================
# AUTO-DETECT DATASET
# ================================================================
file_path = None
for root, _, files in os.walk("/kaggle/input"):
    for fname in files:
        if "primevul_test_paired" in fname:
            file_path = os.path.join(root, fname)
            break
    if file_path:
        break

if file_path is None:
    raise FileNotFoundError("primevul_test_paired not found in /kaggle/input")
print(f"✅ Dataset : {file_path}")

# ================================================================
# LOAD MODEL
# ----------------------------------------------------------------
# WHY bfloat16 instead of float16?
#   Mistral-7B has weight values that can exceed float16's max
#   (~65504). bfloat16 has the same 16-bit footprint but covers
#   the same range as float32, eliminating NaN/Inf spikes that
#   silently corrupt logit scores on Kaggle T4s.
# ================================================================
print(f"⏳ Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Mistral tokenizer ships without a pad token — must set manually
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("✅ Model loaded\n")

# ================================================================
# PRE-CACHE YES / NO TOKEN IDs
# ----------------------------------------------------------------
# Mistral uses SentencePiece BPE. "Yes" and "No" are single tokens
# in its vocabulary. We cache them once here and reuse in scoring.
# ================================================================
YES_IDS = tokenizer.encode("Yes", add_special_tokens=False)
NO_IDS  = tokenizer.encode("No",  add_special_tokens=False)
YES_TOK = YES_IDS[0]
NO_TOK  = NO_IDS[0]
print(f"   YES token id : {YES_TOK}  → '{tokenizer.decode([YES_TOK])}'")
print(f"   NO  token id : {NO_TOK}   → '{tokenizer.decode([NO_TOK])}'\n")

# ================================================================
# CODE PREPROCESSOR
# ----------------------------------------------------------------
# PrimeVul C/C++ functions often start with 60-200 tokens of
# #include chains, #ifdef guards, and Doxygen-style comments before
# any logic appears. Stripping them lets the model focus on the
# actual code logic within the 512-token budget.
# ================================================================
_ML_COMMENT = re.compile(r'/\*.*?\*/', re.DOTALL)
_SL_COMMENT = re.compile(r'//[^\n]*')
_DIRECTIVE  = re.compile(r'^\s*#[^\n]*', re.MULTILINE)
_BLANK3     = re.compile(r'\n{3,}')

def preprocess(code: str) -> str:
    code = _ML_COMMENT.sub('', code)
    code = _SL_COMMENT.sub('', code)
    code = _DIRECTIVE.sub('', code)
    code = _BLANK3.sub('\n\n', code)
    return code.strip()

# ================================================================
# PROMPT BUILDER — Mistral-Instruct format
# ----------------------------------------------------------------
# CRITICAL: Mistral-Instruct-v0.1 was fine-tuned on:
#   <s>[INST] {user_message} [/INST]
#
# It does NOT use <<SYS>> blocks (that is Llama-2/CodeLlama only).
# Using <<SYS>> with Mistral causes the model to echo the tag
# literally and degrades instruction-following significantly.
#
# WHY YES/NO instead of SAFE/VULNERABLE?
#   Any word present in the prompt has elevated probability in the
#   model's next-token distribution. With both label words in the
#   prompt the model oscillates between them based on recency bias,
#   producing near-random outputs. YES/NO carry no such bias.
# ================================================================
SYSTEM = (
    "You are an expert C/C++ security auditor specialising in "
    "memory safety bugs, integer overflows, format string "
    "vulnerabilities, and injection flaws. "
    "You respond with exactly one word: Yes or No."
)

def build_prompt(code: str) -> str:
    code = preprocess(code)

    # Hard-truncate at token level to avoid CUDA OOM
    toks = tokenizer.encode(code, add_special_tokens=False)
    if len(toks) > MAX_CODE_TOKS:
        code = tokenizer.decode(toks[:MAX_CODE_TOKS], skip_special_tokens=True)
        code += "\n// [truncated]"

    # Mistral-Instruct template: system message is prepended inside [INST]
    return (
        f"<s>[INST] {SYSTEM}\n\n"
        f"Examine the following C/C++ function carefully. "
        f"Does it contain a security vulnerability such as a buffer overflow, "
        f"use-after-free, integer overflow, format string bug, or injection flaw?\n\n"
        f"```c\n{code}\n```\n\n"
        f"Reply with exactly one word — Yes if vulnerable, No if safe. [/INST]"
    )

# ================================================================
# LABEL EXTRACTOR (text-mode fallback)
# Only used when LOGIT_MODE=False or logit scoring fails.
# Scans only newly generated tokens; first matching line wins.
# ================================================================
_YES_RE = re.compile(r'\byes\b', re.IGNORECASE)
_NO_RE  = re.compile(r'\bno\b',  re.IGNORECASE)

def extract_yn(raw: str) -> int:
    """1 = vulnerable (Yes), 0 = safe (No), -1 = unparseable."""
    for line in raw.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        y = _YES_RE.search(line)
        n = _NO_RE.search(line)
        if y and n:
            return 1 if y.start() < n.start() else 0
        if y:
            return 1
        if n:
            return 0
    return -1

# ================================================================
# GENERATION HELPERS
# ================================================================
def _tokenize(prompt: str):
    return tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
        padding=False,
    ).to(model.device)

def _generate_text(inputs, temperature: float = 0.0) -> str:
    kwargs = dict(
        max_new_tokens=MAX_NEW_TOKS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if temperature == 0.0:
        kwargs["do_sample"] = False
    else:
        kwargs["do_sample"]  = True
        kwargs["temperature"] = temperature
        kwargs["top_p"]       = 0.92

    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)

    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True)

def _score_logits(inputs) -> float | None:
    """
    Compute log P(Yes) − log P(No) at the first generated position.

    WHY THIS IS BETTER THAN FREE-TEXT DECODING:
      • Uses the full softmax distribution, not just argmax.
      • The score is a calibrated confidence value you can threshold.
      • Completely deterministic — no sampling variance.
      • Works even when the model generates a verbose response,
        because we only look at the very first token's distribution.

    Returns None on any exception (OOM, NaN, shape mismatch).
    """
    try:
        with torch.no_grad():
            logits = model(**inputs).logits      # (1, seq_len, vocab)
        next_logits = logits[0, -1, :]           # distribution over token t+1
        log_probs   = torch.log_softmax(next_logits, dim=-1)
        score = (log_probs[YES_TOK] - log_probs[NO_TOK]).item()
        return score
    except Exception as e:
        print(f"   [logit score error: {e}]")
        return None

# ================================================================
# MAIN PREDICT FUNCTION
# ================================================================
def predict(code: str) -> tuple[int, dict]:
    """
    Returns (label, metadata).
    label : 1 = vulnerable, 0 = safe, -1 = skip
    """
    prompt = build_prompt(code)
    inputs = _tokenize(prompt)
    meta   = {"mode": "?", "score": None, "votes": [], "raw": ""}

    # ── LOGIT MODE (preferred) ──────────────────────────────────
    if LOGIT_MODE:
        score = _score_logits(inputs)
        if score is not None:
            meta["mode"]  = "logit"
            meta["score"] = round(score, 4)
            return (1 if score > LOGIT_THRESH else 0), meta

    # ── MAJORITY-VOTE FALLBACK ──────────────────────────────────
    votes, last_raw = [], ""
    for i in range(VOTE_ROUNDS):
        temp     = 0.0 if i == 0 else VOTE_TEMP
        raw      = _generate_text(inputs, temperature=temp)
        last_raw = raw
        v = extract_yn(raw)
        if v != -1:
            votes.append(v)

    meta.update({"mode": "vote", "votes": votes, "raw": last_raw[:120]})

    if not votes:
        return -1, meta

    return (1 if sum(votes) > len(votes) / 2 else 0), meta

# ================================================================
# LOAD DATASET
# ================================================================
dataset = []
with open(file_path) as f:
    for line in f:
        line = line.strip()
        if line:
            dataset.append(json.loads(line))

if N_SAMPLES:
    dataset = dataset[:N_SAMPLES]

print(f"✅ Loaded {len(dataset)} samples")

# ================================================================
# EVALUATION LOOP
# ================================================================
y_true, y_pred = [], []
skipped  = 0
log_rows = []

print("\n🚀 Starting evaluation...\n")

for i, sample in enumerate(tqdm(dataset)):
    code  = sample["func"]
    label = int(sample["target"])

    pred, meta = predict(code)

    log_rows.append({
        "index":      i,
        "true_label": label,
        "pred_label": pred,
        "mode":       meta["mode"],
        "score":      meta.get("score"),
        "votes":      str(meta.get("votes", [])),
        "raw":        meta.get("raw", "")[:100],
        "skipped":    pred == -1,
    })

    if pred == -1:
        skipped += 1
    else:
        y_true.append(label)
        y_pred.append(pred)

    if (i + 1) % BATCH_SAVE == 0:
        pd.DataFrame(log_rows).to_csv(
            f"{OUT_DIR}/mistral_per_sample.csv", index=False
        )

pd.DataFrame(log_rows).to_csv(f"{OUT_DIR}/mistral_per_sample.csv", index=False)

# ================================================================
# METRICS
# ================================================================
total = len(dataset)
print(f"\n{'='*55}")
print(f"  Total   : {total}")
print(f"  Valid   : {len(y_pred)}")
print(f"  Skipped : {skipped}  ({100*skipped/total:.1f}%)")
print(f"{'='*55}")

if not y_pred:
    print("\n❌ No valid predictions — run diagnostics below.")
else:
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)
    cm   = confusion_matrix(y_true, y_pred)

    results = {
        "accuracy":  round(acc,  4),
        "precision": round(prec, 4),
        "recall":    round(rec,  4),
        "f1":        round(f1,   4),
        "mcc":       round(mcc,  4),
        "n_valid":   len(y_pred),
        "n_skipped": skipped,
    }

    print("\n🚀 FINAL RESULTS")
    print(f"   accuracy  : {acc:.4f}")
    print(f"   precision : {prec:.4f}")
    print(f"   recall    : {rec:.4f}")
    print(f"   f1        : {f1:.4f}")
    print(f"   mcc       : {mcc:.4f}   ← primary metric")

    print(f"\n📉 Confusion matrix  (rows=true, cols=pred)")
    print(f"   {cm}")
    print(f"   TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")

    pred_pos = sum(y_pred) / len(y_pred)
    true_pos = sum(y_true) / len(y_true)
    print(f"\n   Pred positive rate : {pred_pos:.3f}")
    print(f"   True positive rate : {true_pos:.3f}")

    if abs(pred_pos - 0.5) < 0.06:
        print("   ⚠  Still ~50/50 — check token IDs and prompt template")
    else:
        print("   ✅ Model is discriminating")

    if LOGIT_MODE:
        scores = [r["score"] for r in log_rows if r["score"] is not None]
        if scores:
            import statistics as stats
            print(f"\n   Logit score stats :")
            print(f"   mean={stats.mean(scores):.3f}  "
                  f"stdev={stats.stdev(scores):.3f}  "
                  f"min={min(scores):.3f}  "
                  f"max={max(scores):.3f}")
            print(f"   (current threshold={LOGIT_THRESH}; "
                  f"run threshold sweep below if MCC is low)")

    pd.DataFrame([results]).to_csv(
        f"{OUT_DIR}/mistral_primevul_results.csv", index=False
    )
    print(f"\n✅ Saved → {OUT_DIR}/")

# ================================================================
# DIAGNOSTICS  (run these cells manually if MCC stays near 0)
# ================================================================
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DIAGNOSTICS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Verify token IDs decoded correctly:
   >>> print(tokenizer.decode([YES_TOK]), tokenizer.decode([NO_TOK]))
   Expected output: Yes  No

2. Inspect raw outputs (vote mode):
   >>> for r in log_rows[:5]: print(r['raw'])

3. Print the rendered prompt for sample 0:
   >>> print(build_prompt(dataset[0]['func']))

4. Check logit score distribution:
   >>> scores = [r['score'] for r in log_rows if r['score']]
   >>> print(f"range: {min(scores):.2f} to {max(scores):.2f}")
   If range < 1.0 the model has no signal — try v0.2 or Mistral-7B-v0.3.

5. Threshold sweep — find the MCC-maximising cut-off:
   >>> import numpy as np
   >>> from sklearn.metrics import matthews_corrcoef
   >>> valid = [(r['score'], r['true_label'])
   ...          for r in log_rows if r['score'] is not None]
   >>> scores_v, trues_v = zip(*valid)
   >>> best_t, best_mcc = 0.0, -1.0
   >>> for t in np.linspace(-4, 4, 81):
   ...     preds = [1 if s > t else 0 for s in scores_v]
   ...     m = matthews_corrcoef(list(trues_v), preds)
   ...     if m > best_mcc: best_t, best_mcc = t, m
   >>> print(f"Best threshold: {best_t:.2f}  MCC: {best_mcc:.4f}")
   Then set LOGIT_THRESH = best_t and re-run full evaluation.

6. Model upgrade path (if 7B hits a ceiling):
   mistralai/Mistral-7B-Instruct-v0.2  ← same template, better RLHF
   mistralai/Mistral-7B-Instruct-v0.3  ← tokenizer improvements
   mistralai/Mixtral-8x7B-Instruct-v0.1 ← MoE, needs ~2x VRAM
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

✅ Dataset : /kaggle/input/datasets/nikunjnawal009/primevul-1mistral/primevul_test_paired.jsonl
⏳ Loading mistralai/Mistral-7B-Instruct-v0.1 ...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model loaded

   YES token id : 5592  → 'Yes'
   NO  token id : 1770   → 'No'

✅ Loaded 870 samples

🚀 Starting evaluation...




100%|██████████| 870/870 [57:04<00:00,  3.94s/it]


  Total   : 870
  Valid   : 870
  Skipped : 0  (0.0%)

🚀 FINAL RESULTS
   accuracy  : 0.5011
   precision : 0.5029
   recall    : 0.2000
   f1        : 0.2862
   mcc       : 0.0029   ← primary metric

📉 Confusion matrix  (rows=true, cols=pred)
   [[349  86]
 [348  87]]
   TN=349  FP=86  FN=348  TP=87

   Pred positive rate : 0.199
   True positive rate : 0.500
   ✅ Model is discriminating

   Logit score stats :
   mean=-0.249  stdev=0.385  min=-2.062  max=0.438
   (current threshold=0.0; run threshold sweep below if MCC is low)

✅ Saved → /kaggle/working/

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DIAGNOSTICS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Verify token IDs decoded correctly:
   >>> print(tokenizer.decode([YES_TOK]), tokenizer.decode([NO_TOK]))
   Expected output: Yes  No

2. Inspect raw outputs (vote mode):
   >>> for r in log_rows[:5]: print(r['raw'])

3. Print the rendered prompt for sample 0:
   >>> print(build_prompt(dataset[0]['fun